# Personalized LOOCV with SHAP Feature-Count Ablation

This notebook evaluates SVM, LDA, and XGBoost independently for EEG-only,
ET-only, and Hybrid EEG+ET data.

A personalized Leave-One-Product-Out Cross-Validation scheme is used:
for every subject, one product window is held out while all remaining
product windows from the same subject are used for training.

## SHAP Feature-Count Ablation

For every LOOCV fold and every model/modality combination:

1. The held-out product is separated from the training products.
2. StandardScaler is fitted using training products only.
3. SHAP feature importance is computed using training products only.
4. Features are ranked according to mean absolute SHAP importance.
5. The same SHAP ranking is used to evaluate:
   - Top-50 features
   - Top-75 features
   - Top-125 features
6. The classifier is trained separately for each feature-count condition.
7. The held-out product is predicted only after all training-dependent
   operations have been completed.

The SHAP ranking is therefore computed once per LOOCV fold/model/modality
and reused for the three feature-count conditions.

## Leakage boundary

For every fold, the held-out product is never used for:

- StandardScaler fitting
- SHAP model fitting
- SHAP feature ranking
- feature-count selection
- classifier fitting

The held-out product is only transformed using the training-fitted scaler
and then passed to the trained classifier.

## Experimental conditions

3 models × 3 modalities × 3 SHAP feature counts = 27 conditions.

Models:
- SVM
- LDA
- XGBoost

Modalities:
- EEG
- ET
- Hybrid

SHAP feature counts:
- 50
- 75
- 125

In [1]:
# Install dependencies if required.
# Run this cell only if the packages are not already installed.

%pip install numpy pandas scikit-learn xgboost shap

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
# =============================================================================
# IMPORTS AND EXPERIMENT CONFIGURATION
# =============================================================================

from pathlib import Path
from collections import Counter
import json
import warnings

import numpy as np
import pandas as pd
import shap

from xgboost import XGBClassifier

from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    cohen_kappa_score,
)
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC


warnings.filterwarnings(
    'ignore',
    category=UserWarning
)


# =============================================================================
# PATHS
# =============================================================================

DATA_DIR = Path(
    r'C:\Users\tiet\Desktop\Paramveer\Dataset\Separate'
)

# Use a NEW result directory so that the existing Top-100 results
# are not overwritten.
RESULTS_DIR = (
    Path.cwd()
    / 'personalized_loocv_shap_ablation_results'
)

RESULTS_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# =============================================================================
# SHAP FEATURE-COUNT ABLATION
# =============================================================================

# These are the only feature-count conditions in this experiment.
TOP_N_VALUES = [50, 75, 125]

# SHAP ranking is computed once up to the largest requested value.
MAX_TOP_N = max(TOP_N_VALUES)


# =============================================================================
# REPRODUCIBILITY
# =============================================================================

RANDOM_STATE = 42


# =============================================================================
# SUBJECT SELECTION
# =============================================================================

# Set to a list such as ['S01', 'S02'] for a quick pilot.
#
# Set to None to run all subjects.
SUBJECTS_TO_RUN = None


print(f'Data: {DATA_DIR}')
print(f'Outputs: {RESULTS_DIR}')
print(f'SHAP feature counts: {TOP_N_VALUES}')
print(f'Maximum SHAP ranking required: {MAX_TOP_N}')

c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Data: C:\Users\tiet\Desktop\Paramveer\Dataset\Separate
Outputs: c:\Users\tiet\Downloads\personalized_loocv_shap_ablation_results
SHAP feature counts: [50, 75, 125]
Maximum SHAP ranking required: 125


In [3]:
# =============================================================================
# LOAD SUBJECT-WISE FINAL FEATURES
# =============================================================================
#
# This cell only loads the already extracted/synchronized final feature
# representations.
#
# It does NOT:
#   - alter synchronization
#   - alter feature ordering
#   - perform preprocessing
#   - perform feature extraction
#   - perform IMF decomposition
#   - modify labels
#
# The EEG, ET and y arrays are already final products of the extraction
# pipeline.
# =============================================================================


def discover_subjects(data_dir):
    """
    Discover subject IDs from subjects.txt when available.

    Otherwise discover subjects from *_eeg_X.npy files.
    """

    subject_file = data_dir / 'subjects.txt'

    if subject_file.exists():

        return [
            line.strip()
            for line in subject_file.read_text().splitlines()
            if line.strip()
        ]

    return sorted(
        path.name.replace(
            '_eeg_X.npy',
            ''
        )
        for path in data_dir.glob(
            '*_eeg_X.npy'
        )
    )


def load_subjects(
    data_dir,
    selected_subjects=None
):
    """
    Load final subject-wise EEG, ET and label arrays.

    Returns
    -------
    dict
        {
            subject_id: {
                'eeg': EEG feature matrix,
                'et': ET feature matrix,
                'y': binary labels
            }
        }
    """

    subjects = discover_subjects(
        data_dir
    )

    if selected_subjects is not None:

        requested = set(
            selected_subjects
        )

        subjects = [
            sid
            for sid in subjects
            if sid in requested
        ]

    data = {}

    for sid in subjects:

        eeg = np.load(
            data_dir / f'{sid}_eeg_X.npy'
        ).astype(
            np.float64
        )

        et = np.load(
            data_dir / f'{sid}_et_X.npy'
        ).astype(
            np.float64
        )

        y = np.load(
            data_dir / f'{sid}_y.npy'
        ).astype(
            int
        ).ravel()

        if not (
            len(eeg)
            == len(et)
            == len(y)
        ):
            raise ValueError(
                f'{sid}: EEG, ET, and y '
                'must have identical row counts.'
            )

        data[sid] = {
            'eeg': eeg,
            'et': et,
            'y': y
        }

    return data


# =============================================================================
# LOAD DATA
# =============================================================================

subject_data = load_subjects(
    DATA_DIR,
    SUBJECTS_TO_RUN
)

if not subject_data:
    raise RuntimeError(
        'No subject files were found.'
    )


# =============================================================================
# POOLED AUDIT INDEX
# =============================================================================
#
# These arrays preserve subject identity for auditing only.
# They are NEVER supplied to the classifiers.
# =============================================================================

pooled_subject_id = np.concatenate(
    [
        np.repeat(
            sid,
            len(item['y'])
        )
        for sid, item
        in subject_data.items()
    ]
)

pooled_y = np.concatenate(
    [
        item['y']
        for item in subject_data.values()
    ]
)


# =============================================================================
# DATA SUMMARY
# =============================================================================

summary = pd.DataFrame(
    [
        {
            'subject_id': sid,
            'n_products': len(item['y']),
            'buy': int(item['y'].sum()),
            'no_buy': int(
                (item['y'] == 0).sum()
            ),
            'eeg_columns': item['eeg'].shape[1],
            'et_columns': item['et'].shape[1]
        }
        for sid, item
        in subject_data.items()
    ]
)


display(summary)

print(
    f'{len(summary)} subjects; '
    f'{len(pooled_y)} product windows.'
)

,subject_id,n_products,buy,no_buy,eeg_columns,et_columns
0,S01,79,12,67,1330,10
1,S02,97,15,82,1330,10
2,S03,85,15,70,1330,10
3,S05,103,8,95,1330,10
4,S06,134,23,111,1330,10
5,S07,74,12,62,1330,10
6,S08,115,13,102,1330,10
7,S09,86,19,67,1330,10
8,S10,79,15,64,1330,10
9,S12,105,20,85,1330,10


42 subjects; 4265 product windows.


In [4]:
# =============================================================================
# FEATURE METADATA
# =============================================================================
#
# Stable human-readable feature IDs are constructed so that selected
# features can be interpreted and compared across folds/models.
#
# EEG:
#   EEG|channel|IMF|feature
#
# ET:
#   ET|feature
#
# Hybrid:
#   EEG metadata followed by ET metadata.
# =============================================================================


CHANNELS = [
    'Fp1',
    'Fp2',
    'F3',
    'F4',
    'C3',
    'C4',
    'P3',
    'P4',
    'O1',
    'O2',
    'F7',
    'F8',
    'T7',
    'T8',
    'P7',
    'P8',
    'Fz',
    'Cz',
    'Pz'
]


# =============================================================================
# FEATURE NAME CONFIGURATION
# =============================================================================

eeg_config = json.loads(
    (
        DATA_DIR
        / 'feature_names_eeg.json'
    ).read_text()
)

et_config = json.loads(
    (
        DATA_DIR
        / 'feature_names_et.json'
    ).read_text()
)


EEG_FEATURES = eeg_config[
    'features'
]

ET_FEATURES = et_config[
    'features'
]


# =============================================================================
# EEG METADATA
# =============================================================================

def build_eeg_metadata(
    n_columns
):
    """
    Build human-readable metadata for EEG columns.
    """

    rows = []

    per_imf = len(
        EEG_FEATURES
    )

    imfs_per_channel = (
        n_columns
        // (
            len(CHANNELS)
            * per_imf
        )
    )

    for column in range(
        n_columns
    ):

        channel_index = (
            column
            // (
                imfs_per_channel
                * per_imf
            )
        )

        remainder = (
            column
            % (
                imfs_per_channel
                * per_imf
            )
        )

        imf_index = (
            remainder
            // per_imf
        )

        feature_index = (
            remainder
            % per_imf
        )

        feature_name = (
            EEG_FEATURES[
                feature_index
            ]
        )

        channel = (
            CHANNELS[
                channel_index
            ]
        )

        rows.append(
            {
                'feature_id':
                    f'EEG|{channel}|'
                    f'IMF{imf_index + 1}|'
                    f'{feature_name}',

                'source_modality':
                    'EEG',

                'source_column':
                    column,

                'channel':
                    channel,

                'imf':
                    imf_index + 1,

                'feature_name':
                    feature_name
            }
        )

    return pd.DataFrame(
        rows
    )


# =============================================================================
# ET METADATA
# =============================================================================

def build_et_metadata(
    n_columns
):
    """
    Build human-readable metadata for ET columns.
    """

    return pd.DataFrame(
        [
            {
                'feature_id':
                    f'ET|{ET_FEATURES[i]}',

                'source_modality':
                    'ET',

                'source_column':
                    i,

                'channel':
                    None,

                'imf':
                    None,

                'feature_name':
                    ET_FEATURES[i]
            }
            for i in range(
                n_columns
            )
        ]
    )


# =============================================================================
# BUILD MODALITY METADATA
# =============================================================================

n_eeg = next(
    iter(
        subject_data.values()
    )
)['eeg'].shape[1]

n_et = next(
    iter(
        subject_data.values()
    )
)['et'].shape[1]


metadata_by_modality = {

    'EEG':
        build_eeg_metadata(
            n_eeg
        ),

    'ET':
        build_et_metadata(
            n_et
        )
}


metadata_by_modality[
    'Hybrid'
] = pd.concat(
    [
        metadata_by_modality[
            'EEG'
        ],
        metadata_by_modality[
            'ET'
        ]
    ],
    ignore_index=True
)


# Stable model-column index
for metadata in (
    metadata_by_modality.values()
):

    metadata[
        'model_column'
    ] = np.arange(
        len(metadata)
    )


display(
    metadata_by_modality[
        'Hybrid'
    ].head()
)

,feature_id,source_modality,source_column,channel,imf,feature_name,model_column
0,EEG|Fp1|IMF1|relative_energy,EEG,0,Fp1,1,relative_energy,0
1,EEG|Fp1|IMF1|differential_entropy,EEG,1,Fp1,1,differential_entropy,1
2,EEG|Fp1|IMF1|sample_entropy,EEG,2,Fp1,1,sample_entropy,2
3,EEG|Fp1|IMF1|higuchi_fd,EEG,3,Fp1,1,higuchi_fd,3
4,EEG|Fp1|IMF1|katz_fd,EEG,4,Fp1,1,katz_fd,4


In [5]:
# =============================================================================
# CLASSIFIER FACTORIES AND SHAP-BASED FEATURE RANKING
# =============================================================================
#
# The SHAP selector uses the same classifier family as the eventual evaluator.
#
# IMPORTANT:
# The selector is fitted only on the current LOOCV training fold.
#
# No test product is supplied to the SHAP selector.
# =============================================================================


def make_model(
    model_name
):

    if model_name == 'SVM':

        # Linear SVM mirrors the base-paper classifier
        # and permits linear SHAP explanations.

        return SVC(
            kernel='linear',
            C=1.0,
            probability=False,
            random_state=RANDOM_STATE
        )


    if model_name == 'LDA':

        # Shrinkage keeps LDA numerically stable
        # when feature count is large relative to samples.

        return LinearDiscriminantAnalysis(
            solver='lsqr',
            shrinkage='auto'
        )


    if model_name == 'XGB':

        return XGBClassifier(
            n_estimators=150,
            max_depth=3,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            objective='binary:logistic',
            eval_metric='logloss',
            random_state=RANDOM_STATE,
            n_jobs=1,
            verbosity=0
        )


    raise ValueError(
        f'Unknown model: {model_name}'
    )


# =============================================================================
# POSITIVE-CLASS SCORE
# =============================================================================

def positive_score(
    model,
    X
):

    if hasattr(
        model,
        'predict_proba'
    ):

        class_index = list(
            model.classes_
        ).index(1)

        return model.predict_proba(
            X
        )[:, class_index]

    return model.decision_function(
        X
    )


# =============================================================================
# NORMALIZE SHAP OUTPUT
# =============================================================================

def normalise_shap_values(
    values
):

    if isinstance(
        values,
        list
    ):

        values = values[-1]

    if hasattr(
        values,
        'values'
    ):

        values = values.values

    values = np.asarray(
        values
    )

    if values.ndim == 3:

        values = values[
            :,
            :,
            -1
        ]

    if values.ndim != 2:

        raise ValueError(
            f'Unexpected SHAP array shape: '
            f'{values.shape}'
        )

    return values


# =============================================================================
# SHAP FEATURE RANKING
# =============================================================================

def select_top_features_with_shap(
    model_name,
    X_train_scaled,
    y_train,
    top_n
):
    """
    Fit the model on the current training fold and rank features using
    mean absolute SHAP importance.

    Parameters
    ----------
    model_name : str
        SVM, LDA or XGB.

    X_train_scaled : ndarray
        Scaled training data only.

    y_train : ndarray
        Training labels only.

    top_n : int
        Maximum number of ranked features to return.

    Returns
    -------
    selected : ndarray
        Feature indices sorted from highest to lowest SHAP importance.

    selected_importance : ndarray
        Corresponding SHAP importances.

    selector : fitted model
        SHAP selector model.
    """

    selector = make_model(
        model_name
    )

    selector.fit(
        X_train_scaled,
        y_train
    )


    # -------------------------------------------------------------------------
    # SHAP
    # -------------------------------------------------------------------------

    if model_name == 'XGB':

        shap_values = (
            shap.TreeExplainer(
                selector
            ).shap_values(
                X_train_scaled
            )
        )

    else:

        shap_values = (
            shap.LinearExplainer(
                selector,
                X_train_scaled
            ).shap_values(
                X_train_scaled
            )
        )


    # -------------------------------------------------------------------------
    # Mean absolute SHAP importance
    # -------------------------------------------------------------------------

    importance = np.mean(
        np.abs(
            normalise_shap_values(
                shap_values
            )
        ),
        axis=0
    )


    # -------------------------------------------------------------------------
    # Complete ranking up to top_n
    # -------------------------------------------------------------------------

    n_available = (
        X_train_scaled.shape[1]
    )

    n_select = min(
        int(top_n),
        n_available
    )

    ranked_features = np.argsort(
        importance
    )[::-1]

    selected = ranked_features[
        :n_select
    ]

    selected_importance = (
        importance[
            selected
        ]
    )


    return (
        selected.astype(int),
        selected_importance,
        selector
    )

In [6]:
# =============================================================================
# PERSONALIZED LOOCV WITH SHAP FEATURE-COUNT ABLATION
# =============================================================================
#
# For every subject:
#   For every modality:
#       For every held-out product:
#           1. Split into training and test product.
#           2. Fit StandardScaler on training products only.
#           3. For every model:
#                  a. Fit SHAP selector on training data only.
#                  b. Generate one ranking up to MAX_TOP_N.
#                  c. Evaluate Top-50.
#                  d. Evaluate Top-75.
#                  e. Evaluate Top-125.
#
# IMPORTANT:
# SHAP is calculated ONCE per fold/model/modality.
#
# The same SHAP ranking is reused for all three feature-count conditions.
#
# Therefore the ablation changes ONLY the number of retained features.
# =============================================================================


MODELS = [
    'SVM',
    'LDA',
    'XGB'
]

MODALITIES = [
    'EEG',
    'ET',
    'Hybrid'
]


# =============================================================================
# MODALITY MATRIX
# =============================================================================

def get_modality_matrix(
    subject,
    modality
):

    if modality == 'EEG':

        return subject[
            'eeg'
        ]


    if modality == 'ET':

        return subject[
            'et'
        ]


    if modality == 'Hybrid':

        return np.column_stack(
            [
                subject['eeg'],
                subject['et']
            ]
        )


    raise ValueError(
        modality
    )


# =============================================================================
# PERSONALIZED LOOCV
# =============================================================================

def run_personalized_loocv(
    subject_data
):

    prediction_rows = []
    selection_rows = []


    # =========================================================================
    # SUBJECT LOOP
    # =========================================================================

    for (
        subject_number,
        (sid, subject)
    ) in enumerate(
        subject_data.items(),
        start=1
    ):

        y = subject[
            'y'
        ]

        print(
            f'[{subject_number}/'
            f'{len(subject_data)}] '
            f'{sid}: '
            f'{len(y)} product windows'
        )


        # =====================================================================
        # MODALITY LOOP
        # =====================================================================

        for modality in MODALITIES:

            X = get_modality_matrix(
                subject,
                modality
            )

            feature_metadata = (
                metadata_by_modality[
                    modality
                ]
            )


            # =================================================================
            # LOOCV LOOP
            # =================================================================

            for held_out_index in range(
                len(y)
            ):

                # -------------------------------------------------------------
                # Train/test split
                # -------------------------------------------------------------

                train_mask = np.ones(
                    len(y),
                    dtype=bool
                )

                train_mask[
                    held_out_index
                ] = False


                X_train_raw = X[
                    train_mask
                ]

                X_test_raw = X[
                    ~train_mask
                ]


                y_train = y[
                    train_mask
                ]

                y_test = y[
                    ~train_mask
                ]


                # -------------------------------------------------------------
                # Standardization
                # -------------------------------------------------------------
                #
                # FIT ONLY ON TRAINING DATA.
                # -------------------------------------------------------------

                scaler = StandardScaler()

                X_train_scaled = (
                    scaler.fit_transform(
                        X_train_raw
                    )
                )

                X_test_scaled = (
                    scaler.transform(
                        X_test_raw
                    )
                )


                # =============================================================
                # MODEL LOOP
                # =============================================================

                for model_name in MODELS:


                    # ---------------------------------------------------------
                    # One-class training fold
                    # ---------------------------------------------------------

                    if np.unique(
                        y_train
                    ).size < 2:

                        majority = int(
                            y_train[0]
                        )


                        # Record every ablation condition explicitly.
                        for top_n in TOP_N_VALUES:

                            prediction_rows.append(
                                {
                                    'subject_id':
                                        sid,

                                    'held_out_product_index':
                                        held_out_index,

                                    'modality':
                                        modality,

                                    'model':
                                        model_name,

                                    'top_n':
                                        top_n,

                                    'y_true':
                                        int(
                                            y_test[0]
                                        ),

                                    'y_pred':
                                        majority,

                                    'score':
                                        np.nan,

                                    'status':
                                        'single_class_training_fold'
                                }
                            )

                        continue


                    # =========================================================
                    # SHAP RANKING
                    # =========================================================
                    #
                    # THIS IS THE CRITICAL ABLATION CHANGE.
                    #
                    # SHAP is calculated ONCE using only this fold's training
                    # data and ranked up to MAX_TOP_N = 125.
                    #
                    # The same ranking is then reused for Top-50, Top-75 and
                    # Top-125.
                    # =========================================================

                    ranked_features, ranked_importance, _ = (
                        select_top_features_with_shap(
                            model_name,
                            X_train_scaled,
                            y_train,
                            MAX_TOP_N
                        )
                    )


                    # =========================================================
                    # FEATURE-COUNT ABLATION
                    # =========================================================

                    for top_n in TOP_N_VALUES:


                        # -----------------------------------------------------
                        # Take prefix of the SAME SHAP ranking
                        # -----------------------------------------------------

                        n_select = min(
                            top_n,
                            len(ranked_features)
                        )


                        selected = (
                            ranked_features[
                                :n_select
                            ]
                        )

                        selected_importance = (
                            ranked_importance[
                                :n_select
                            ]
                        )


                        # -----------------------------------------------------
                        # Metadata
                        # -----------------------------------------------------

                        selected_metadata = (
                            feature_metadata.iloc[
                                selected
                            ]
                        )


                        # -----------------------------------------------------
                        # Store feature selection information
                        # -----------------------------------------------------

                        selection_rows.append(
                            {
                                'subject_id':
                                    sid,

                                'held_out_product_index':
                                    held_out_index,

                                'modality':
                                    modality,

                                'model':
                                    model_name,

                                'top_n':
                                    top_n,

                                'n_selected':
                                    len(
                                        selected
                                    ),

                                'feature_ids_json':
                                    json.dumps(
                                        selected_metadata[
                                            'feature_id'
                                        ].tolist()
                                    ),

                                'model_columns_json':
                                    json.dumps(
                                        selected.tolist()
                                    ),

                                'mean_abs_shap_json':
                                    json.dumps(
                                        [
                                            float(x)
                                            for x
                                            in selected_importance
                                        ]
                                    )
                            }
                        )


                        # =====================================================
                        # TRAIN FINAL EVALUATOR
                        # =====================================================

                        evaluator = make_model(
                            model_name
                        )


                        evaluator.fit(
                            X_train_scaled[
                                :,
                                selected
                            ],
                            y_train
                        )


                        # =====================================================
                        # TEST PREDICTION
                        # =====================================================

                        X_test_selected = (
                            X_test_scaled[
                                :,
                                selected
                            ]
                        )


                        y_pred = int(
                            evaluator.predict(
                                X_test_selected
                            )[0]
                        )


                        score = float(
                            positive_score(
                                evaluator,
                                X_test_selected
                            )[0]
                        )


                        # =====================================================
                        # STORE PREDICTION
                        # =====================================================

                        prediction_rows.append(
                            {
                                'subject_id':
                                    sid,

                                'held_out_product_index':
                                    held_out_index,

                                'modality':
                                    modality,

                                'model':
                                    model_name,

                                'top_n':
                                    top_n,

                                'y_true':
                                    int(
                                        y_test[0]
                                    ),

                                'y_pred':
                                    y_pred,

                                'score':
                                    score,

                                'status':
                                    'ok'
                            }
                        )


    # =========================================================================
    # RETURN RESULTS
    # =========================================================================

    return (
        pd.DataFrame(
            prediction_rows
        ),
        pd.DataFrame(
            selection_rows
        )
    )

In [7]:
# =============================================================================
# RUN THE COMPLETE SHAP FEATURE-COUNT ABLATION
# =============================================================================
#
# One notebook execution produces all:
#
#   Top-50
#   Top-75
#   Top-125
#
# for all:
#
#   3 models × 3 modalities
#
# No separate notebooks/files are required.
# =============================================================================


predictions, selected_feature_lists = (
    run_personalized_loocv(
        subject_data
    )
)


# =============================================================================
# SAVE FOLD-LEVEL RESULTS
# =============================================================================

predictions.to_csv(
    RESULTS_DIR
    / 'fold_predictions_shap_ablation.csv',
    index=False
)


selected_feature_lists.to_csv(
    RESULTS_DIR
    / 'fold_selected_feature_lists_shap_ablation.csv',
    index=False
)


print(
    'Saved fold-level predictions '
    'and SHAP feature-selection results.'
)

print(
    f'Feature-count conditions: '
    f'{TOP_N_VALUES}'
)

print(
    f'Prediction rows: '
    f'{len(predictions)}'
)

print(
    f'Selection rows: '
    f'{len(selected_feature_lists)}'
)

[1/42] S01: 79 product windows


c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_bas

[2/42] S02: 97 product windows


c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_bas

[3/42] S03: 85 product windows


c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_bas

... (75 outputs left out)

c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_bas

[42/42] S44: 111 product windows


c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
c:\Users\tiet\AppData\Local\Programs\Python\Python314\Lib\site-packages\sklearn\svm\_base.py:239: FutureWarning: The `probability` parameter was deprecated in 1.9 and will be removed in version 1.11. Use `CalibratedClassifierCV(SVC(), ensemble=False)` instead of `SVC(probability=True)`
  warnings.warn(
Background dataset has 110 samples but max_samples=100. Subsampling to 100 samples for S

Saved fold-level predictions and SHAP feature-selection results.
Feature-count conditions: [50, 75, 125]
Prediction rows: 115155
Selection rows: 115155


In [8]:
# =============================================================================
# PER-SUBJECT METRICS AND AVERAGES
# =============================================================================
#
# Metrics are calculated independently for:
#
#   top_n × modality × model
#
# Therefore Top-50, Top-75 and Top-125 are never mixed.
# =============================================================================


def metric_row(
    frame
):

    y_true = frame[
        'y_true'
    ].to_numpy()

    y_pred = frame[
        'y_pred'
    ].to_numpy()


    usable_scores = (
        frame[
            'score'
        ].notna()
    )


    # ---------------------------------------------------------
    # AUC
    # ---------------------------------------------------------

    auc = np.nan

    if (
        len(
            np.unique(
                y_true
            )
        ) == 2
        and usable_scores.all()
    ):

        auc = roc_auc_score(
            y_true,
            frame[
                'score'
            ].to_numpy()
        )


    # ---------------------------------------------------------
    # Metrics
    # ---------------------------------------------------------

    return {

        'n_products':
            len(frame),

        'accuracy':
            accuracy_score(
                y_true,
                y_pred
            ),

        'precision':
            precision_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'recall':
            recall_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'f1':
            f1_score(
                y_true,
                y_pred,
                zero_division=0
            ),

        'auc_roc':
            auc,

        'cohen_kappa':
            cohen_kappa_score(
                y_true,
                y_pred
            ),

        'nonstandard_folds':
            int(
                (
                    frame[
                        'status'
                    ]
                    != 'ok'
                ).sum()
            )
    }


# =============================================================================
# PER-SUBJECT METRICS
# =============================================================================

per_subject_metric_rows = []


for (
    subject_id,
    modality,
    model,
    top_n
), frame in predictions.groupby(
    [
        'subject_id',
        'modality',
        'model',
        'top_n'
    ],
    sort=True
):

    row = metric_row(
        frame
    )


    row.update(
        {
            'subject_id':
                subject_id,

            'modality':
                modality,

            'model':
                model,

            'top_n':
                top_n
        }
    )


    per_subject_metric_rows.append(
        row
    )


per_subject_metrics = pd.DataFrame(
    per_subject_metric_rows
)


# =============================================================================
# AVERAGE SUBJECT METRICS
# =============================================================================
#
# IMPORTANT:
# We first compute metrics separately for every subject.
# Then we average those subject-level metrics.
#
# This preserves the personalized evaluation structure.
# =============================================================================

average_metrics = (
    per_subject_metrics
    .groupby(
        [
            'top_n',
            'modality',
            'model'
        ],
        sort=True
    )[
        [
            'accuracy',
            'precision',
            'recall',
            'f1',
            'auc_roc',
            'cohen_kappa'
        ]
    ]
    .mean()
    .reset_index()
)


# =============================================================================
# EXPORT
# =============================================================================

per_subject_metrics.to_csv(
    RESULTS_DIR
    / 'per_subject_metrics_shap_ablation.csv',
    index=False
)


average_metrics.to_csv(
    RESULTS_DIR
    / 'average_subject_metrics_shap_ablation.csv',
    index=False
)


# =============================================================================
# DISPLAY
# =============================================================================

display(
    average_metrics.sort_values(
        [
            'top_n',
            'model',
            'modality'
        ]
    )
)

,top_n,modality,model,accuracy,precision,recall,f1,auc_roc,cohen_kappa
0,50,EEG,LDA,0.807970,0.452616,0.459450,0.454223,0.735792,0.334035
3,50,ET,LDA,0.852553,0.593285,0.470401,0.515815,0.843123,0.432130
6,50,Hybrid,LDA,0.809579,0.457897,0.455024,0.454047,0.742880,0.335076
1,50,EEG,SVM,0.805026,0.455849,0.424026,0.437186,0.769758,0.314466
4,50,ET,SVM,0.846609,0.504472,0.333799,0.387363,0.795495,0.319995
7,50,Hybrid,SVM,0.798411,0.447268,0.408176,0.424438,0.771910,0.296901
2,50,EEG,XGB,0.841210,0.543976,0.367674,0.423566,0.827735,0.339605
5,50,ET,XGB,0.837188,0.519641,0.418494,0.460181,0.811343,0.365191
8,50,Hybrid,XGB,0.838155,0.519839,0.359386,0.411773,0.825283,0.326760
9,75,EEG,LDA,0.816915,0.474453,0.450654,0.459639,0.736068,0.345347


In [9]:
# =============================================================================
# FEATURE COMMONALITY ANALYSIS FOR SHAP ABLATION
# =============================================================================
#
# Within-model:
#   Features present in EVERY successful LOOCV selection set for a given:
#
#       top_n × modality × model
#
# Across-models:
#   Intersection of SVM, LDA and XGB within-model common feature sets
#   for the same:
#
#       top_n × modality
#
# Selection frequency:
#   Number and proportion of successful LOOCV folds in which each feature
#   was selected.
#
# This keeps Top-50, Top-75 and Top-125 completely independent.
# =============================================================================


def metadata_lookup(
    modality
):

    return (
        metadata_by_modality[
            modality
        ]
        .set_index(
            'feature_id'
        )
    )


frequency_rows = []
common_within_rows = []

# Key:
#   (top_n, modality, model)
common_sets = {}


# =============================================================================
# 1. WITHIN-MODEL FEATURE FREQUENCY AND COMMON FEATURES
# =============================================================================

for (
    top_n,
    modality,
    model
), group in selected_feature_lists.groupby(
    [
        'top_n',
        'modality',
        'model'
    ],
    sort=True
):


    # -------------------------------------------------------------------------
    # Selected feature IDs from every successful fold
    # -------------------------------------------------------------------------

    fold_sets = [
        set(
            json.loads(
                value
            )
        )
        for value
        in group[
            'feature_ids_json'
        ]
    ]


    # -------------------------------------------------------------------------
    # Strict intersection:
    # feature must occur in EVERY successful fold
    # -------------------------------------------------------------------------

    common_set = (
        set.intersection(
            *fold_sets
        )
        if fold_sets
        else set()
    )


    common_sets[
        (
            top_n,
            modality,
            model
        )
    ] = common_set


    # -------------------------------------------------------------------------
    # Frequency of each feature
    # -------------------------------------------------------------------------

    counts = Counter(
        feature_id
        for fold_set
        in fold_sets
        for feature_id
        in fold_set
    )


    lookup = metadata_lookup(
        modality
    )


    # -------------------------------------------------------------------------
    # Selection-frequency table
    # -------------------------------------------------------------------------

    for feature_id, count in counts.items():

        row = lookup.loc[
            feature_id
        ].to_dict()


        # feature_id is the lookup index,
        # so restore it explicitly.

        row[
            'feature_id'
        ] = feature_id


        row.update(
            {
                'top_n':
                    top_n,

                'modality':
                    modality,

                'model':
                    model,

                'selection_count':
                    count,

                'n_successful_folds':
                    len(fold_sets),

                'selection_rate':
                    (
                        count / len(fold_sets)
                        if len(fold_sets) > 0
                        else 0.0
                    )
            }
        )


        frequency_rows.append(
            row
        )


    # -------------------------------------------------------------------------
    # Strict within-model common features
    # -------------------------------------------------------------------------

    for feature_id in sorted(
        common_set
    ):

        row = lookup.loc[
            feature_id
        ].to_dict()


        row[
            'feature_id'
        ] = feature_id


        row.update(
            {
                'top_n':
                    top_n,

                'modality':
                    modality,

                'model':
                    model,

                'n_folds_intersected':
                    len(fold_sets)
            }
        )


        common_within_rows.append(
            row
        )


# =============================================================================
# 2. SELECTION FREQUENCY DATAFRAME
# =============================================================================

if frequency_rows:

    selection_frequency = (
        pd.DataFrame(
            frequency_rows
        )
        .sort_values(
            [
                'top_n',
                'modality',
                'model',
                'selection_rate',
                'feature_id'
            ],
            ascending=[
                True,
                True,
                True,
                False,
                True
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    selection_frequency = pd.DataFrame(
        columns=[
            'feature_id',
            'top_n',
            'modality',
            'model',
            'selection_count',
            'n_successful_folds',
            'selection_rate'
        ]
    )


# =============================================================================
# 3. WITHIN-MODEL COMMON FEATURE DATAFRAME
# =============================================================================

if common_within_rows:

    common_within_model = (
        pd.DataFrame(
            common_within_rows
        )
        .sort_values(
            [
                'top_n',
                'modality',
                'model',
                'feature_id'
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    common_within_model = pd.DataFrame(
        columns=[
            'feature_id',
            'top_n',
            'modality',
            'model',
            'n_folds_intersected'
        ]
    )


# =============================================================================
# 4. ACROSS-MODEL COMMON FEATURES
# =============================================================================
#
# For each feature count and modality:
#
#     SVM common features
#             ∩
#     LDA common features
#             ∩
#     XGB common features
#
# =============================================================================

common_across_rows = []


for top_n in TOP_N_VALUES:

    for modality in MODALITIES:


        model_common_sets = [
            common_sets.get(
                (
                    top_n,
                    modality,
                    model
                ),
                set()
            )
            for model
            in MODELS
        ]


        if model_common_sets:

            shared = set.intersection(
                *model_common_sets
            )

        else:

            shared = set()


        lookup = metadata_lookup(
            modality
        )


        for feature_id in sorted(
            shared
        ):

            row = lookup.loc[
                feature_id
            ].to_dict()


            row[
                'feature_id'
            ] = feature_id


            row.update(
                {
                    'top_n':
                        top_n,

                    'modality':
                        modality,

                    'models_intersected':
                        '|'.join(
                            MODELS
                        )
                }
            )


            common_across_rows.append(
                row
            )


# =============================================================================
# 5. ACROSS-MODEL DATAFRAME
# =============================================================================

if common_across_rows:

    common_across_models = (
        pd.DataFrame(
            common_across_rows
        )
        .sort_values(
            [
                'top_n',
                'modality',
                'feature_id'
            ]
        )
        .reset_index(
            drop=True
        )
    )

else:

    common_across_models = pd.DataFrame(
        columns=[
            'feature_id',
            'top_n',
            'modality',
            'models_intersected'
        ]
    )


# =============================================================================
# 6. EXPORT
# =============================================================================

selection_frequency.to_csv(
    RESULTS_DIR
    / 'feature_selection_frequency_shap_ablation.csv',
    index=False
)


common_within_model.to_csv(
    RESULTS_DIR
    / 'common_features_within_model_shap_ablation.csv',
    index=False
)


common_across_models.to_csv(
    RESULTS_DIR
    / 'common_features_across_models_shap_ablation.csv',
    index=False
)


# =============================================================================
# 7. DISPLAY
# =============================================================================

print(
    'Common-feature exports saved.'
)

print(
    f'\nSelection-frequency rows: '
    f'{len(selection_frequency)}'
)

print(
    f'Within-model common features: '
    f'{len(common_within_model)}'
)

print(
    f'Across-model common features: '
    f'{len(common_across_models)}'
)


display(
    common_within_model
)

display(
    common_across_models
)

Common-feature exports saved.

Selection-frequency rows: 20003
Within-model common features: 90
Across-model common features: 30


,source_modality,source_column,channel,imf,feature_name,model_column,feature_id,top_n,modality,model,n_folds_intersected
0,ET,9,None,None,dwell_time_ms,9,ET|dwell_time_ms,50,ET,LDA,4265
1,ET,0,None,None,fixation_count,0,ET|fixation_count,50,ET,LDA,4265
2,ET,1,None,None,mean_fixation_duration_ms,1,ET|mean_fixation_duration_ms,50,ET,LDA,4265
3,ET,7,None,None,mean_pupil_diameter_px,7,ET|mean_pupil_diameter_px,50,ET,LDA,4265
4,ET,4,None,None,mean_saccade_amplitude_px,4,ET|mean_saccade_amplitude_px,50,ET,LDA,4265
...,...,...,...,...,...,...,...,...,...,...,...
85,ET,5,None,None,mean_saccade_velocity_px_s,5,ET|mean_saccade_velocity_px_s,125,ET,XGB,4265
86,ET,8,None,None,pupil_diameter_variability_px,8,ET|pupil_diameter_variability_px,125,ET,XGB,4265
87,ET,6,None,None,scanpath_length_px,6,ET|scanpath_length_px,125,ET,XGB,4265
88,ET,3,None,None,time_to_first_fixation_ms,3,ET|time_to_first_fixation_ms,125,ET,XGB,4265


,source_modality,source_column,channel,imf,feature_name,model_column,feature_id,top_n,modality,models_intersected
0,ET,9,None,None,dwell_time_ms,9,ET|dwell_time_ms,50,ET,SVM|LDA|XGB
1,ET,0,None,None,fixation_count,0,ET|fixation_count,50,ET,SVM|LDA|XGB
2,ET,1,None,None,mean_fixation_duration_ms,1,ET|mean_fixation_duration_ms,50,ET,SVM|LDA|XGB
3,ET,7,None,None,mean_pupil_diameter_px,7,ET|mean_pupil_diameter_px,50,ET,SVM|LDA|XGB
4,ET,4,None,None,mean_saccade_amplitude_px,4,ET|mean_saccade_amplitude_px,50,ET,SVM|LDA|XGB
5,ET,5,None,None,mean_saccade_velocity_px_s,5,ET|mean_saccade_velocity_px_s,50,ET,SVM|LDA|XGB
6,ET,8,None,None,pupil_diameter_variability_px,8,ET|pupil_diameter_variability_px,50,ET,SVM|LDA|XGB
7,ET,6,None,None,scanpath_length_px,6,ET|scanpath_length_px,50,ET,SVM|LDA|XGB
8,ET,3,None,None,time_to_first_fixation_ms,3,ET|time_to_first_fixation_ms,50,ET,SVM|LDA|XGB
9,ET,2,None,None,total_fixation_duration_ms,2,ET|total_fixation_duration_ms,50,ET,SVM|LDA|XGB


## Output files

All three SHAP feature-count experiments are produced by this single notebook.

### Fold-level outputs

- `fold_predictions_shap_ablation.csv`
  - One held-out prediction for every subject, product window,
    modality, model and SHAP feature-count condition.

- `fold_selected_feature_lists_shap_ablation.csv`
  - SHAP-selected feature lists for every successful LOOCV fold.
  - Contains `top_n` to distinguish Top-50, Top-75 and Top-125.

### Evaluation outputs

- `per_subject_metrics_shap_ablation.csv`
  - Subject-level metrics for every:
    - SHAP feature count
    - modality
    - model

- `average_subject_metrics_shap_ablation.csv`
  - Mean subject-level metrics across the study population.

### Feature-analysis outputs

- `feature_selection_frequency_shap_ablation.csv`
  - Frequency and selection rate of features across successful LOOCV folds.

- `common_features_within_model_shap_ablation.csv`
  - Features appearing in every successful LOOCV selection set for:
    `top_n × modality × model`.

- `common_features_across_models_shap_ablation.csv`
  - Features shared by SVM, LDA and XGB for:
    `top_n × modality`.

### Experimental conditions

The complete experiment contains:

3 SHAP feature counts
× 3 modalities
× 3 models
= 27 conditions.

SHAP feature counts:
- Top-50
- Top-75
- Top-125

Modalities:
- EEG
- ET
- Hybrid

Models:
- SVM
- LDA
- XGB

### SHAP ranking rule

For each LOOCV fold, SHAP is fitted using the training products only.
Features are ranked by mean absolute SHAP value.

The same ranking is reused for all three feature-count conditions:

SHAP ranking
    ├── Top-50
    ├── Top-75
    └── Top-125

Therefore, the ablation changes only the number of retained features.